In [ ]:
import torch
from esm.models.esmc import EsmcForMaskedLM, EsmcTokenizer

In [20]:
import torch.nn.functional as F

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ESMC-300M Pretrained model
model = EsmcForMaskedLM.from_pretrained(
    "biohub/ESMC-300M",
    device=device,
).eval()

tokenizer = EsmcTokenizer()

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/home/nisanga/projects/huh-dna-specificity/.venv/lib/python3.12/site-packages/esm/models/esmc/config.py:206: FutureWarning: config.json uses pre-alignment ESMC field names ['d_model', 'n_heads', 'n_layers']; the current names are ['hidden_size', 'num_attention_heads', 'num_hidden_layers'].
  return cls.from_dict(json.load(f))


In [18]:
def get_protein_embedding(amino_acid_seq):

    inputs = tokenizer([amino_acid_seq], return_tensors="pt", padding=True)
    inputs = {name: value.to(model.device) for name, value in inputs.items()}

    with torch.inference_mode():
        result = model(**inputs, output_hidden_states=True)

    token_embeddings = result.hidden_states[-1]    
    residue_embeddings = token_embeddings[0, 1:-1] # exlude start and end token

    return residue_embeddings


In [21]:
pcv_seq = "MPSKKNGRSGPQPHKRWVFTLNNPSEDERKKIREPPISLFDYFIVGEEGNEEGRTPHLQGFANFVKKQTFNKVKWYLGARCHIEKAKGTDQQNKEYCSKEGNLLIECGAPRSQGQRSDLSTAVSTLLESGSLVTVAEQHPVTFVRNFRGLAELLKVSGKMQKRDWKTNVHVIVGPPGCGKSKWAANFADPETTYWKPPRNKWWDGYHGEEVVVIDDFYGWLPWDDLLRLCDRYPLTVETKGGTVPFLARSILITSNQTPLEWYSSTAVPAVEALYRRITSLVFWKNATEQSTEEGGQFVTLSPPCPEFPYEINY"
wdv_seq = "MASSSAPRFRVYSKYLFLTYPECTLEPQYALDSLRTLLNKYEPLYIAAVRELHEDGSPHLHVLVQNKLRASITNPNALNLRMDTSPFSIFHPNIQAAKDCNQVRDYITKEVDSDVNTAEWGTFVAVSTPGRKDRDADMKQIIESSSSREEFLSMVCNRFPFEWSIRLKDFEYTARHLFPDPVATYTPEFPTESLICHETIESWKNEHLYSESPGRHKSIYICGPTRTGKTSWARSLGTHNYYNSLVDFTTYDVNAKYNIIDDIPFKFTPNWKCFVGAQRDFTVNPKYGKRKVIRGGIPCIILVNPDEDWLKDMTPEQSDYMYSNTVVHYMYEGETFINYSFASGEDVTASQ"
fbnyv_seq = "MARQVICWCFTLNNPLSPLSLHDSMKYLVYQTEQGEAGNIHFQGYIEMKKRTSLAGMKKLIPGAHFEKRRGTQGEARAYSMKEDTRLEGPWEYGEFVPTIEDKLREVMNDMKITGKRPIEYIEECCNTYDKSASTLREFRGELKKKKAISSWELQRKPWMGEVDALLQERDGRRIIWVYGPQGGEGKTSYAKHLVKTRDAFYSTGGKTADIAFAWDHQELVLFDFPRSFEEYVNYGVIEQLKNGIIQSGKYQSVIKYSDYVEVIVFANFTPRSGMFSEDRIVYVYA"

pcv_embed = get_protein_embedding(pcv_seq)
wdv_embed = get_protein_embedding(wdv_seq)
fbnyv_embed = get_protein_embedding(fbnyv_seq)

wdv_sDBM = wdv_embed[90:99, :]
pcv_sDBM = pcv_embed[78:87, :]
fbnyv_sDBM = fbnyv_embed[62:71, :]


In [49]:
def query_sDBM(protein_seq, known_sDBM):

    protein_embed = get_protein_embedding(protein_seq)

    length, _ = protein_embed.shape

    max_mean = -1
    next_max_mean = -1

    start_seq = 0
    end_seq = 0 

    for i in range(30, 150):
        match_block = protein_embed[i:i+9, :]

        sim = F.cosine_similarity(match_block, known_sDBM, dim = 1)

        mean = torch.mean(sim)

        # print(i + 1)
        # print(mean)

        if mean > max_mean:
            start_seq = i + 1
            end_seq = i + 9
            next_max_mean = max_mean
            max_mean = mean
        elif mean > next_max_mean:
            next_max_mean = mean
            next_start = i + 1
            next_end = i + 9

    if ((max_mean - next_max_mean) < 0.01):
        print("Unable to suggest sDBM... confidence measure below threshold")
        return

  #  diff = ((max_mean - next_max_mean) / max_mean) * 100
    perc = (max_mean - next_max_mean) * 100


    print("sDBM candidate found!!!")
    print(f"Max Mean Similarity: {max_mean}")
    print(f"Closest Mean Similarity to sDBM Prediction: {next_max_mean}")
    print(f"sDBM prediction from {start_seq} to {end_seq}")
    print(f"{next_start} to {next_end}")
    print(protein_seq[start_seq - 1: end_seq])
    print(perc)

In [50]:
query_sDBM(pcv_seq, wdv_sDBM)
query_sDBM(fbnyv_seq, pcv_sDBM)
query_sDBM(fbnyv_seq, wdv_sDBM)

sDBM candidate found!!!
Max Mean Similarity: 0.6590912938117981
Closest Mean Similarity to sDBM Prediction: 0.5100229382514954
sDBM prediction from 80 to 88
44 to 52
RCHIEKAKG
tensor(14.9068, device='cuda:0')
sDBM candidate found!!!
Max Mean Similarity: 0.7518791556358337
Closest Mean Similarity to sDBM Prediction: 0.4692811071872711
sDBM prediction from 62 to 70
44 to 52
PGAHFEKRR
tensor(28.2598, device='cuda:0')
sDBM candidate found!!!
Max Mean Similarity: 0.5649231672286987
Closest Mean Similarity to sDBM Prediction: 0.5359272360801697
sDBM prediction from 63 to 71
93 to 101
GAHFEKRRG
tensor(2.8996, device='cuda:0')
